# Hotel Booking Dataset — Executive EDA

Download the hotel booking CSV from the LMS **Study Material** tab, then run the one code cell below in Google Colab. It creates a cleaned dataset and a management-ready EDA report based on the uploaded data.

In [ ]:
# End-to-end Hotel Booking EDA — one Google Colab cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('Please upload the hotel booking CSV file.')

df = pd.read_csv(csv_files[0])
df.columns = df.columns.str.strip().str.replace(r'\s+', '_', regex=True)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='deep')
original_shape = df.shape
original_missing = df.isna().sum()
original_duplicates = df.duplicated().sum()

def find_column(candidates):
    normalized = {col.lower().replace('_', ' ').replace('-', ' ').strip(): col for col in df.columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    for name, original in normalized.items():
        if any(candidate in name for candidate in candidates):
            return original
    return None

hotel_col = find_column(['hotel', 'hotel type'])
cancel_col = find_column(['is canceled', 'is_cancelled', 'canceled', 'cancelled', 'booking status'])
lead_col = find_column(['lead time'])
adr_col = find_column(['adr', 'average daily rate', 'daily rate'])
market_col = find_column(['market segment', 'segment'])
channel_col = find_column(['distribution channel', 'booking channel', 'channel'])
country_col = find_column(['country', 'guest country'])
meal_col = find_column(['meal', 'meal plan'])
month_col = find_column(['arrival date month', 'arrival month', 'month'])
year_col = find_column(['arrival date year', 'arrival year', 'year'])
weekend_col = find_column(['stays in weekend nights', 'weekend nights'])
week_col = find_column(['stays in week nights', 'week nights'])
adults_col = find_column(['adults'])
children_col = find_column(['children'])
babies_col = find_column(['babies'])
requests_col = find_column(['total of special requests', 'special requests'])

print('=' * 90)
print(f'HOTEL BOOKING DATASET — END-TO-END EDA: {csv_files[0]}')
print('=' * 90)
print(f'Original dataset: {original_shape[0]:,} rows × {original_shape[1]} columns')
display(df.head())
print('\n1. DATA UNDERSTANDING AND QUALITY ASSESSMENT')
print('Original data types:')
display(df.dtypes.to_frame(name='Data Type'))
print('Missing values before cleaning:')
display(pd.DataFrame({'Missing': original_missing, 'Missing (%)': (original_missing / len(df) * 100).round(2)}).sort_values('Missing', ascending=False))
print(f'Duplicate records before cleaning: {original_duplicates:,}')

# CLEANING AND PREPROCESSING
cleaning_steps = []
# Standardise blank-like text values.
for column in df.select_dtypes(include='object').columns:
    df[column] = df[column].astype(str).str.strip().replace({'': np.nan, 'nan': np.nan, 'None': np.nan, 'NULL': np.nan})
cleaning_steps.append('Trimmed text fields and standardized blank-like values as missing.')
# Remove duplicate records.
df = df.drop_duplicates().copy()
cleaning_steps.append(f'Removed {original_duplicates:,} duplicate records.')
# Numerical type correction and median imputation.
numeric_keywords = ['lead', 'stay', 'adult', 'child', 'baby', 'week', 'day', 'adr', 'rate', 'request', 'parking', 'waiting', 'previous', 'booking changes']
for column in df.columns:
    if any(word in column.lower() for word in numeric_keywords):
        converted = pd.to_numeric(df[column], errors='coerce')
        if converted.notna().sum() >= max(1, df[column].notna().sum() * 0.5):
            df[column] = converted
for column in df.select_dtypes(include=np.number).columns:
    if df[column].isna().any():
        df[column] = df[column].fillna(df[column].median())
cleaning_steps.append('Corrected usable numeric fields and imputed their missing values with the median.')
# Categorical imputation uses mode; country uses an explicit Unknown category.
for column in df.select_dtypes(include='object').columns:
    if df[column].isna().any():
        if column == country_col:
            df[column] = df[column].fillna('Unknown')
        elif not df[column].mode(dropna=True).empty:
            df[column] = df[column].fillna(df[column].mode(dropna=True)[0])
cleaning_steps.append('Imputed categorical fields with their mode, while retaining unknown guest country explicitly.')
# Derive booking measures.
if weekend_col and week_col:
    df['Total_Nights'] = df[weekend_col].fillna(0) + df[week_col].fillna(0)
if adults_col:
    df['Total_Guests'] = df[adults_col].fillna(0)
    for column in [children_col, babies_col]:
        if column:
            df['Total_Guests'] += df[column].fillna(0)
    invalid_guest_rows = (df['Total_Guests'] == 0).sum()
    df = df[df['Total_Guests'] > 0].copy()
    cleaning_steps.append(f'Removed {invalid_guest_rows:,} records with zero guests.')
# IQR capping handles extreme numeric values without deleting valid booking records.
outlier_columns = [column for column in [lead_col, adr_col, 'Total_Nights'] if column in df.columns]
outlier_log = []
for column in outlier_columns:
    q1, q3 = df[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr > 0:
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outliers = ((df[column] < lower) | (df[column] > upper)).sum()
        df[column] = df[column].clip(lower, upper)
        outlier_log.append(f'{column}: capped {outliers:,} IQR outlier(s)')
if outlier_log:
    cleaning_steps.append('Applied IQR-based capping to extreme values: ' + '; '.join(outlier_log) + '.')

# Normalize cancellation indicator where the source uses text labels.
if cancel_col:
    if not pd.api.types.is_numeric_dtype(df[cancel_col]):
        df[cancel_col] = df[cancel_col].astype(str).str.lower().map({'1': 1, 'yes': 1, 'cancelled': 1, 'canceled': 1, '0': 0, 'no': 0, 'not cancelled': 0, 'not canceled': 0})
    df[cancel_col] = pd.to_numeric(df[cancel_col], errors='coerce').fillna(0).astype(int)

print('\n2. CLEANED DATASET VERIFICATION')
print(f'Cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('Remaining missing values:')
display(df.isna().sum().to_frame(name='Missing').query('Missing > 0'))
display(df.describe(include='all').T)

# UNIVARIATE ANALYSIS
print('\n3. UNIVARIATE ANALYSIS')
if adr_col:
    plt.figure(figsize=(9, 5))
    sns.histplot(df[adr_col].dropna(), bins=35, kde=True, color='#2978b5')
    plt.title('Distribution of Average Daily Rate (ADR)')
    plt.xlabel('ADR')
    plt.ylabel('Number of Bookings')
    plt.tight_layout()
    plt.show()
if hotel_col and cancel_col:
    plt.figure(figsize=(8, 5))
    cancel_rate_hotel = df.groupby(hotel_col)[cancel_col].mean().sort_values(ascending=False) * 100
    sns.barplot(x=cancel_rate_hotel.index, y=cancel_rate_hotel.values, hue=cancel_rate_hotel.index, legend=False, palette='Reds_r')
    plt.title('Cancellation Rate by Hotel Type')
    plt.xlabel('Hotel Type')
    plt.ylabel('Cancellation Rate (%)')
    plt.tight_layout()
    plt.show()

# BIVARIATE AND GROUP-WISE ANALYSIS
print('\n4. BIVARIATE AND GROUP-WISE ANALYSIS')
if lead_col and cancel_col:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x=cancel_col, y=lead_col, hue=cancel_col, legend=False, palette='Set2')
    plt.title('Lead Time by Cancellation Outcome')
    plt.xlabel('Cancelled (0 = No, 1 = Yes)')
    plt.ylabel('Lead Time')
    plt.tight_layout()
    plt.show()
if month_col and cancel_col:
    month_order = ['January','February','March','April','May','June','July','August','September','October','November','December']
    monthly = df.groupby(month_col)[cancel_col].agg(['size', 'mean']).reindex(month_order).dropna(how='all')
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.plot(monthly.index, monthly['size'], marker='o', color='#1f77b4', label='Bookings')
    ax1.set_xlabel('Arrival Month')
    ax1.set_ylabel('Bookings', color='#1f77b4')
    ax2 = ax1.twinx()
    ax2.plot(monthly.index, monthly['mean'] * 100, marker='s', color='#d62728', label='Cancellation Rate')
    ax2.set_ylabel('Cancellation Rate (%)', color='#d62728')
    plt.title('Bookings and Cancellation Rate by Arrival Month')
    plt.xticks(rotation=45, ha='right')
    fig.tight_layout()
    plt.show()
else:
    monthly = None
if market_col and cancel_col:
    market_summary = df.groupby(market_col).agg(Bookings=(cancel_col, 'size'), Cancellation_Rate=(cancel_col, 'mean')).sort_values('Bookings', ascending=False)
    market_summary['Cancellation_Rate'] = (market_summary['Cancellation_Rate'] * 100).round(1)
    print('Market segment summary:')
    display(market_summary)
else:
    market_summary = None
if hotel_col and adr_col:
    hotel_summary = df.groupby(hotel_col).agg(Bookings=(adr_col, 'size'), Average_ADR=(adr_col, 'mean'))
    if cancel_col:
        hotel_summary['Cancellation_Rate'] = df.groupby(hotel_col)[cancel_col].mean() * 100
    print('Hotel type summary:')
    display(hotel_summary.round(2))
else:
    hotel_summary = None

# CORRELATION ANALYSIS
print('\n5. CORRELATION ANALYSIS')
numeric_data = df.select_dtypes(include=np.number)
if numeric_data.shape[1] >= 2:
    correlation = numeric_data.corr()
    plt.figure(figsize=(max(10, numeric_data.shape[1] * 0.8), max(7, numeric_data.shape[1] * 0.6)))
    sns.heatmap(correlation, cmap='coolwarm', center=0, annot=False, square=True)
    plt.title('Correlation Heatmap of Numerical Booking Variables')
    plt.tight_layout()
    plt.show()
    pairs = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool)).stack()
    strongest_pair = pairs.abs().idxmax() if not pairs.empty else None
    strongest_corr = pairs.loc[strongest_pair] if strongest_pair else None
else:
    strongest_pair, strongest_corr = None, None

# EXECUTIVE INSIGHTS AND RECOMMENDATIONS
print('\n' + '=' * 90)
print('EXECUTIVE INSIGHTS')
print('=' * 90)
insights = []
if cancel_col:
    insights.append(f'Overall cancellation rate is {df[cancel_col].mean() * 100:.1f}% across {len(df):,} cleaned bookings.')
if hotel_col and cancel_col:
    worst_hotel = cancel_rate_hotel.index[0]
    insights.append(f'{worst_hotel} has the highest cancellation rate at {cancel_rate_hotel.iloc[0]:.1f}%.')
if lead_col and cancel_col:
    lead_by_outcome = df.groupby(cancel_col)[lead_col].mean()
    insights.append(f'Cancelled bookings average {lead_by_outcome.get(1, np.nan):.1f} days of lead time versus {lead_by_outcome.get(0, np.nan):.1f} for retained bookings.')
if hotel_summary is not None:
    top_adr_hotel = hotel_summary['Average_ADR'].idxmax()
    insights.append(f'{top_adr_hotel} achieves the higher average ADR ({hotel_summary.loc[top_adr_hotel, "Average_ADR"]:.2f}).')
if monthly is not None and not monthly.empty:
    peak_month = monthly['size'].idxmax()
    insights.append(f'{peak_month} is the highest-volume arrival month in the dataset ({monthly.loc[peak_month, "size"]:,.0f} bookings).')
if market_summary is not None and not market_summary.empty:
    biggest_segment = market_summary.index[0]
    insights.append(f'{biggest_segment} is the largest market segment with {market_summary.iloc[0]["Bookings"]:,.0f} bookings.')
if strongest_pair:
    insights.append(f'The strongest numerical relationship is between {strongest_pair[0]} and {strongest_pair[1]} (correlation {strongest_corr:.2f}).')
for number, insight in enumerate(insights[:5], start=1):
    print(f'{number}. {insight}')

print('\nMANAGEMENT RECOMMENDATIONS')
recommendations = [
    'Tighten cancellation controls for high-risk segments through deposits, flexible rebooking options, or reminder communications.',
    'Use lead-time segmentation to target early bookers with confirmation nudges and conversion offers.',
    'Adjust rate and inventory strategy around peak booking months to protect ADR while capturing demand.',
    'Prioritize the highest-volume market segments with tailored packages and channel-specific retention campaigns.',
    'Review hotel-type differences in cancellation and ADR to align pricing, overbooking, and staffing policies.',
    'Track the strongest correlated operating metrics as leading indicators in a regular management dashboard.',
    'Maintain data-quality controls for duplicate bookings, zero-guest records, and missing profile fields.'
]
for number, recommendation in enumerate(recommendations[:7], start=1):
    print(f'{number}. {recommendation}')

# Export deliverables created from the uploaded data.
cleaned_file = 'cleaned_hotel_booking_dataset.csv'
df.to_csv(cleaned_file, index=False)
report_file = 'hotel_booking_executive_eda_report.md'
with open(report_file, 'w', encoding='utf-8') as report:
    report.write('# Hotel Booking Executive EDA Report\n\n')
    report.write(f'**Dataset:** {csv_files[0]}  \n**Cleaned records:** {len(df):,}  \n**Fields:** {len(df.columns)}\n\n')
    report.write('## Data Quality and Preparation\n')
    for step in cleaning_steps:
        report.write(f'- {step}\n')
    report.write('\n## Key Business Insights\n')
    for number, insight in enumerate(insights[:5], start=1):
        report.write(f'{number}. {insight}\n')
    report.write('\n## Management Recommendations\n')
    for number, recommendation in enumerate(recommendations[:7], start=1):
        report.write(f'{number}. {recommendation}\n')
    report.write('\n## Scope\nThe notebook includes descriptive statistics, univariate and bivariate analysis, group-wise comparisons, correlation analysis, and visualizations supporting these conclusions.\n')
print(f'\nSaved: {cleaned_file} and {report_file}')
files.download(cleaned_file)
files.download(report_file)